# 06 FID Evaluation

Generate samples and compute FID against the target dataset.

In [ ]:

from pathlib import Path
import os, json, yaml, math, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from torchvision import datasets, transforms, utils
from torch.utils.data import DataLoader, random_split, Subset

from src.utils.seed import set_seed
from src.models.unet import UNetModel
from src.training.trainer import SDDTrainer
from src.evaluation.feature_extract import extract_features
from src.evaluation.linear_probe import train_linear_probe
from src.evaluation.fid import FIDEvaluator

ROOT = Path('.')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
set_seed(42)


In [ ]:

import yaml, copy, itertools, time
from pathlib import Path

def load_cfg(path):
    return yaml.safe_load(Path(path).read_text())

def make_loaders(cfg):
    size = cfg["dataset"]["image_size"]
    train_tfm = transforms.Compose([
        transforms.Resize((size, size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3),
    ])
    eval_tfm = transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3),
    ])
    if cfg["dataset"]["name"] == "cifar10":
        train_ds = datasets.CIFAR10(root=cfg["dataset"]["root"], train=True, download=True, transform=train_tfm)
        test_ds = datasets.CIFAR10(root=cfg["dataset"]["root"], train=False, download=True, transform=eval_tfm)
    else:
        train_ds = datasets.ImageFolder(root=cfg["dataset"]["train_dir"], transform=train_tfm)
        test_ds = datasets.ImageFolder(root=cfg["dataset"]["val_dir"], transform=eval_tfm)
    train_loader = DataLoader(
        train_ds,
        batch_size=cfg["train"]["batch_size"],
        shuffle=True,
        num_workers=cfg["dataset"]["num_workers"],
        pin_memory=True,
        drop_last=True,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=cfg["train"]["batch_size"],
        shuffle=False,
        num_workers=cfg["dataset"]["num_workers"],
        pin_memory=True,
        drop_last=False,
    )
    return train_loader, test_loader

def build_model(cfg):
    return UNetModel(
        base_channels=cfg["model"]["channels"],
        channel_mults=tuple(cfg["model"]["channel_mults"]),
        num_res_blocks=cfg["model"]["num_res_blocks"],
        attention_resolutions=tuple(cfg["model"]["attention_resolutions"]),
        dropout=cfg["model"]["dropout"],
        image_size=cfg["dataset"]["image_size"],
    )

def make_optimizer(trainer, cfg):
    params = list(trainer.model.parameters())
    if getattr(trainer, "proj_student", None) is not None:
        params += list(trainer.proj_student.parameters())
    return torch.optim.AdamW(params, lr=cfg["train"]["lr"], weight_decay=cfg["train"]["weight_decay"])

def maybe_init_wandb(cfg, name=None):
    if not cfg.get("wandb", {}).get("enabled", False):
        return None
    import wandb
    return wandb.init(
        project=cfg["wandb"]["project"],
        entity=cfg["wandb"].get("entity"),
        name=name or cfg.get("run_name"),
        config=cfg,
        tags=cfg["wandb"].get("tags", []),
        reinit=True,
    )

def save_cfg(cfg, path):
    Path(path).write_text(yaml.safe_dump(cfg, sort_keys=False))

def train_epochs(trainer, loader, cfg, optimizer, run=None, epochs=None, ckpt_dir="outputs/checkpoints"):
    ckpt_dir = Path(ckpt_dir)
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    scaler = torch.cuda.amp.GradScaler(enabled=cfg["train"]["mixed_precision"] and torch.cuda.is_available())
    epochs = epochs or cfg["train"]["epochs"]
    history = []
    for epoch in range(epochs):
        trainer.state.epoch = epoch
        metrics = trainer.train_one_epoch(loader, optimizer, scaler=scaler, wandb_run=run)
        history.append({"epoch": epoch, **metrics})
        if (epoch + 1) % cfg["train"]["save_every"] == 0:
            torch.save({
                "model": trainer.model.state_dict(),
                "teacher": None if trainer.teacher is None else trainer.teacher.state_dict(),
                "proj_student": None if trainer.proj_student is None else trainer.proj_student.state_dict(),
                "proj_teacher": None if trainer.proj_teacher is None else trainer.proj_teacher.state_dict(),
                "epoch": epoch,
                "cfg": cfg,
            }, ckpt_dir / f"{cfg.get('run_name', 'run')}_epoch{epoch+1}.pt")
    return pd.DataFrame(history)


In [ ]:

cfg = load_cfg("outputs/cifar10_sdd_full.yaml") if Path("outputs/cifar10_sdd_full.yaml").exists() else load_cfg("configs/cifar10.yaml")
train_loader, test_loader = make_loaders(cfg)
model = build_model(cfg)
trainer = SDDTrainer(model, cfg, DEVICE)

# Replace with checkpoint loading if available
# ckpt = torch.load("outputs/checkpoints/cifar10_sdd_full_epoch400.pt", map_location=DEVICE)
# trainer.model.load_state_dict(ckpt["model"])

fid = FIDEvaluator(device=DEVICE)
real_batch = next(iter(test_loader))[0].to(DEVICE)[:64]
fid.update_real((real_batch * 0.5 + 0.5).clamp(0, 1))

samples = trainer.sample(n=64, shape=(3, cfg["dataset"]["image_size"], cfg["dataset"]["image_size"]))
fid.update_fake((samples * 0.5 + 0.5).clamp(0, 1))
score = fid.compute()
score


In [ ]:

grid = utils.make_grid(samples[:16].cpu(), nrow=4, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(6,6))
plt.imshow(grid.permute(1,2,0))
plt.axis("off")
plt.show()
